# Hugging Face Transformers 微调训练入门

本示例将介绍基于 Transformers 实现模型微调训练的主要流程，包括：
- 数据集下载
- 数据预处理
- 训练超参数配置
- 训练评估指标设置
- 训练器基本介绍
- 实战训练
- 模型保存

## YelpReviewFull 数据集

**Hugging Face 数据集：[ YelpReviewFull ](https://huggingface.co/datasets/yelp_review_full)**

### 数据集摘要

Yelp评论数据集包括来自Yelp的评论。它是从Yelp Dataset Challenge 2015数据中提取的。

### 支持的任务和排行榜
文本分类、情感分类：该数据集主要用于文本分类：给定文本，预测情感。

### 语言
这些评论主要以英语编写。

### 数据集结构

#### 数据实例
一个典型的数据点包括文本和相应的标签。

来自YelpReviewFull测试集的示例如下：

```json
{
    'label': 0,
    'text': 'I got \'new\' tires from them and within two weeks got a flat. I took my car to a local mechanic to see if i could get the hole patched, but they said the reason I had a flat was because the previous patch had blown - WAIT, WHAT? I just got the tire and never needed to have it patched? This was supposed to be a new tire. \\nI took the tire over to Flynn\'s and they told me that someone punctured my tire, then tried to patch it. So there are resentful tire slashers? I find that very unlikely. After arguing with the guy and telling him that his logic was far fetched he said he\'d give me a new tire \\"this time\\". \\nI will never go back to Flynn\'s b/c of the way this guy treated me and the simple fact that they gave me a used tire!'
}
```

#### 数据字段

- 'text': 评论文本使用双引号（"）转义，任何内部双引号都通过2个双引号（""）转义。换行符使用反斜杠后跟一个 "n" 字符转义，即 "\n"。
- 'label': 对应于评论的分数（介于1和5之间）。

#### 数据拆分

Yelp评论完整星级数据集是通过随机选取每个1到5星评论的130,000个训练样本和10,000个测试样本构建的。总共有650,000个训练样本和50,000个测试样本。

## 下载数据集

In [1]:
%load_ext autotime

time: 55.5 μs (started: 2025-07-20 10:29:30 +08:00)


In [2]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")

/home/wengad/anaconda3/envs/aiup/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


time: 9.72 s (started: 2025-07-20 10:29:31 +08:00)


In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

time: 1.29 ms (started: 2025-07-20 10:30:26 +08:00)


In [4]:
dataset["train"][111]

{'label': 2,
 'text': "As far as Starbucks go, this is a pretty nice one.  The baristas are friendly and while I was here, a lot of regulars must have come in, because they bantered away with almost everyone.  The bathroom was clean and well maintained and the trash wasn't overflowing in the canisters around the store.  The pastries looked fresh, but I didn't partake.  The noise level was also at a nice working level - not too loud, music just barely audible.\\n\\nI do wish there was more seating.  It is nice that this location has a counter at the end of the bar for sole workers, but it doesn't replace more tables.  I'm sure this isn't as much of a problem in the summer when there's the space outside.\\n\\nThere was a treat receipt promo going on, but the barista didn't tell me about it, which I found odd.  Usually when they have promos like that going on, they ask everyone if they want their receipt to come back later in the day to claim whatever the offer is.  Today it was one of th

time: 2.48 ms (started: 2025-07-20 10:30:28 +08:00)


In [5]:
import random
import pandas as pd
import datasets
from IPython.display import display, HTML

time: 301 μs (started: 2025-07-20 10:30:28 +08:00)


In [6]:
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

time: 449 μs (started: 2025-07-20 10:30:29 +08:00)


In [7]:
show_random_elements(dataset["train"])

,label,text
0,1 star,"Very sad!\nMaybe I ordered the wrong item as it is Chinese not Thai but it was pathetic .\nWor won ton $12 for 2 shrimps 2 pieces of pork some chicken and bok Choy I sent it back telling them it had nothing in it , and I got it back w more cabbage . Dishwater\nTea in a mug w a bag!"
1,3 stars,"I would come back for the apps, ambience, friendly staff, and maybe I'd stick to pork or smoked meat for the main. All of the beef burgers here are cooked well done, no choice offered from what we experienced, and that plus something in the grind of the beef left the patty tasting grainy and unappetizing. I went in hungry enough to eat a horse lamp (they have those) and ended up maoing through much of the oversized portion of fries and the burger bun and toppings and everything else around, rather than eat the meat itself. This is a problem in a burger joint, in my humble opinion, but there's certainly lots of other stuff to like about JoBlo."
2,4 stars,Grade: B\n\nVery underrated place. Unique food. Excellent service. Somewhat pricey.
3,1 star,There store employee/manager kristin is not at all friendly. She acts like she's doing a million things instead of providing customer service.
4,1 star,"I've never tried this place before so I went for the first time last night. I ordered a beef dish, fried rice and crab wontons with cream cheese to go. The place was packed, but it had a dirty feel to it and a lot of dirty dishes were stacked in tubs behind the counter. What really disturbed me, though, was that while I was waiting for my food near the register, I had a full view of what was happening in the kitchen since there was no partition. Right in front of the entrance, the cook took out three huge blocks of boneless, frozen chicken parts onto the counter. After sitting they'd thawed a few minutes, I watched him feed them through a deli slicer, into small blocks, presumably to be thawed and cooked up, fat and all. I did not get the impression that the meat was or would be carefully inspected before being cooked. I know a lot of restaurants probably prepare meat that way, but the mass qualities of untrimmed meat I saw being served made me really lose my appetite. When I got home, my wontons smelled really fishy (like they'd been cooked and sat out for a while) and the sauce on my beef was just slimy enough that anything left of hunger quickly fled me. So basically I spent $13 to eat a carton of rice. Their prices are not cheap enough to warrant visiting this place over a quality establishment. Will not be returning."
5,3 stars,"auch ich war schon bei BOC - obwohl ich eigentlich dort den ehemals ans\u00e4ssigen \""Praktiker\"" suchte ;-)\nSp\u00e4ter bin ich sogar extra noch mal hingefahren.\n\n\n Das Sortiment an Fahrr\u00e4dern w\u00fcrde mich bei BOC trotz der Quantit\u00e4t nicht begeistern. Ich habe sie mir deshalb auch nicht genauer angesehen.\n\n\n Ich suchte nach Radkleidung (was bei mir st\u00e4ndig der Fall ist) und wurde in den kleinen Gr\u00f6\u00dfen nur bedingt f\u00fcndig. \nBein n\u00e4chsten Besuch war gerade Ausverkauf der Winterkollektion - (in der unbeheizten Halle) und bei Regenwetter haben wir uns drei Stunden Zeit genommen, nach Schn\u00e4ppchen zu suchen. \nMit mindestens drei neuen (Bekleidungs-)Teilen pro \""Mann\"" zum echten Schn\u00e4ppchenpreis verlie\u00dfen wir BOC. Allerdings haben wir daf\u00fcr auch lange gesucht, in der relaitv unsortierten Masse. \nWar hier Ausverkauf f\u00fcr alle BOC-Online-Shops???\nBeratung hatten wir nicht viel erwartet, und selbst einfache Fragen konnten auch kaum beantwortet werden. Selbst ist die Frau. \nEs gibt dort \u00fcbrigens sowohl Eigenmarken-Artikel als auch Markenartikel von Gore, Shimano etc.\n\n\n Auch nicht schlecht: die Teile-Auswahl. An ZUbeh\u00f6r- und Kleinteilen erh\u00e4lt man hier so ziemlich alles was man sucht- preislich im normalen Rahmen. \nIm Schn\u00e4ppchenmarkt kann man sogar richtiges Gl\u00fcck haben!! \nEs gibt qualitativ hochwer

time: 23.1 ms (started: 2025-07-20 10:30:30 +08:00)


## 预处理数据

下载数据集到本地后，使用 Tokenizer 来处理文本，对于长度不等的输入数据，可以使用填充（padding）和截断（truncation）策略来处理。

Datasets 的 `map` 方法，支持一次性在整个数据集上应用预处理函数。

下面使用填充到最大长度的策略，处理整个数据集：

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

time: 3.61 s (started: 2025-07-20 10:30:32 +08:00)


In [9]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

time: 1.1 ms (started: 2025-07-20 10:30:37 +08:00)


In [10]:
show_random_elements(tokenized_datasets["train"], num_examples=1)

,label,text,input_ids,token_type_ids,attention_mask
0,4 stars,"Someone below mentioned Hondaya in LA, it DOES remind me of this place. We ordered half-size ramen dx, regular shoyu ramen, kushi katsu (tonkatsu on a stick with onions inside), fried quail eggs, beef tongue yakitori, agedashi tofu, honey toast and a pitcher of kirin.\n\nFYI, 30 mins after the place opened, it was completely packed.","[101, 6518, 2071, 3025, 12110, 2315, 1107, 10722, 117, 1122, 141, 19825, 1708, 11484, 1143, 1104, 1142, 1282, 119, 1284, 2802, 1544, 118, 2060, 26084, 1424, 173, 1775, 117, 2366, 188, 5114, 9379, 26084, 1424, 117, 180, 13148, 1182, 24181, 9552, 113, 11371, 24375, 6385, 1113, 170, 6166, 1114, 1113, 5266, 1656, 114, 117, 15688, 186, 6718, 2723, 6471, 117, 14413, 3661, 11078, 2293, 2772, 1182, 117, 4079, 18303, 1106, 14703, 117, 8531, 17458, 1105, 170, 8092, 1104, 180, 17262, 1179, 119, 165, 183, 165, 183, 2271, 3663, 2240, 117, 1476, 11241, 1116, 1170, 1103, 1282, 1533, 117, 1122, 1108, ...]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...]"


time: 7.71 ms (started: 2025-07-20 10:30:38 +08:00)


### 数据抽样

使用 1000 个数据样本，在 BERT 上演示小规模训练（基于 Pytorch Trainer）

`shuffle()`函数会随机重新排列列的值。如果您希望对用于洗牌数据集的算法有更多控制，可以在此函数中指定generator参数来使用不同的numpy.random.Generator。

In [11]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

time: 91.3 ms (started: 2025-07-20 10:30:40 +08:00)


In [12]:
#  使用了全量的数据集做训练和评估
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(649000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

time: 9.9 ms (started: 2025-07-20 10:30:41 +08:00)


## 微调训练配置

### 加载 BERT 模型

警告通知我们正在丢弃一些权重（`vocab_transform` 和 `vocab_layer_norm` 层），并随机初始化其他一些权重（`pre_classifier` 和 `classifier` 层）。在微调模型情况下是绝对正常的，因为我们正在删除用于预训练模型的掩码语言建模任务的头部，并用一个新的头部替换它，对于这个新头部，我们没有预训练的权重，所以库会警告我们在用它进行推理之前应该对这个模型进行微调，而这正是我们要做的事情。

In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


time: 795 ms (started: 2025-07-20 10:30:46 +08:00)


### 训练超参数（TrainingArguments）

完整配置参数与默认值：https://huggingface.co/docs/transformers/v4.36.1/en/main_classes/trainer#transformers.TrainingArguments

源代码定义：https://github.com/huggingface/transformers/blob/v4.36.1/src/transformers/training_args.py#L161

**最重要配置：模型权重保存路径(output_dir)**

In [14]:
from transformers import TrainingArguments

model_dir = "models/bert-base-cased-finetune-yelp"

# logging_steps 默认值为500，根据我们的训练数据和步长，将其设置为100
training_args = TrainingArguments(output_dir=model_dir,
                                  eval_strategy='steps',
                                  per_device_train_batch_size=32,
                                  num_train_epochs=10,
                                  logging_steps=500)

time: 207 ms (started: 2025-07-20 10:35:12 +08:00)


In [15]:
# 完整的超参数配置
print(training_args)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=500,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False,

### 训练过程中的指标评估（Evaluate)

**[Hugging Face Evaluate 库](https://huggingface.co/docs/evaluate/index)** 支持使用一行代码，获得数十种不同领域（自然语言处理、计算机视觉、强化学习等）的评估方法。 当前支持 **完整评估指标：https://huggingface.co/evaluate-metric**

训练器（Trainer）在训练过程中不会自动评估模型性能。因此，我们需要向训练器传递一个函数来计算和报告指标。 

Evaluate库提供了一个简单的准确率函数，您可以使用`evaluate.load`函数加载

In [16]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

time: 2 s (started: 2025-07-20 10:35:34 +08:00)



接着，调用 `compute` 函数来计算预测的准确率。

在将预测传递给 compute 函数之前，我们需要将 logits 转换为预测值（**所有Transformers 模型都返回 logits**）。

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

time: 234 μs (started: 2025-07-20 10:35:38 +08:00)


#### 训练过程指标监控

通常，为了监控训练过程中的评估指标变化，我们可以在`TrainingArguments`指定`evaluation_strategy`参数，以便在 epoch 结束时报告评估指标。

In [21]:
# 教程使用的tf版本是4.37.2，实际操作环境使用4.53的，参数名称不一样，需要使用eval_strategy

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir=model_dir,
                                  evaluation_strategy="epoch", 
                                  per_device_train_batch_size=64,
                                  num_train_epochs=10,
                                  logging_steps=1000)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

time: 71.2 ms (started: 2025-07-20 09:57:35 +08:00)


In [21]:
# 教程使用的tf版本是4.37.2，实际操作环境使用4.53的，参数名称不一样，需要使用eval_strategy

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir=model_dir,
                                  eval_strategy="steps", 
                                  per_device_train_batch_size=64,
                                  num_train_epochs=10,
                                  logging_steps=200)

time: 32 ms (started: 2025-07-20 10:42:38 +08:00)


## 开始训练

### 实例化训练器（Trainer）

`kernel version` 版本问题：暂不影响本示例代码运行

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

time: 7.67 ms (started: 2025-07-20 10:42:41 +08:00)


## 使用 nvidia-smi 查看 GPU 使用

为了实时查看GPU使用情况，可以使用 `watch` 指令实现轮询：`watch -n 2 nvidia-smi`:

```shell
Every 2.0s: nvidia-smi                                                                                                                           mathings: Sun Jul 20 10:41:35 2025

Sun Jul 20 10:41:35 2025
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.158.01             Driver Version: 573.24         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090 ...    On  |   00000000:02:00.0  On |                  N/A |
| N/A   70C    P2            110W /  110W |   12165MiB /  24463MiB |     77%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|    0   N/A  N/A             775      C   /python3.12                           N/A      |
+-----------------------------------------------------------------------------------------+

以上是batchsize=32的

----
Every 2.0s: nvidia-smi                                                                                                                           mathings: Sun Jul 20 11:10:25 2025

Sun Jul 20 11:10:25 2025
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.158.01             Driver Version: 573.24         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090 ...    On  |   00000000:02:00.0  On |                  N/A |
| N/A   72C    P2            101W /  109W |   23417MiB /  24463MiB |     74%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|    0   N/A  N/A             775      C   /python3.12                           N/A      |
+-----------------------------------------------------------------------------------------+
以上是batchsize=64的

```

In [23]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
200,0.790200,0.856099,0.607000
400,0.889000,0.861448,0.628000
600,0.857500,0.791965,0.655000
800,0.856600,0.762962,0.685000
1000,0.834000,0.758281,0.670000
1200,0.825000,0.797542,0.636000
1400,0.806200,0.758490,0.665000
1600,0.807800,0.757421,0.653000
1800,0.794700,0.711307,0.683000
2000,0.803800,0.782212,0.652000


KeyboardInterrupt: 

time: 1d 1h 6min 35s (started: 2025-07-20 10:42:46 +08:00)


跑完两个epoch后，loss有降到0.6以下，acc升到0.715，果然还是要暴力美学
![图片.png](docs/images/train_hist-01.png)

第3个epoch后

![图片.png](docs/images/train_hist-03.png)

第4个epoch后，训练loss下降，val loss上升，acc无明显上升
![图片.png](docs/images/train_hist-04.png)

第4个epoch后，训练loss下降，val loss连续上升，acc无明显上升，直接人工停止掉
![图片.png](docs/images/train_hist-99.png)


In [24]:
small_test_dataset = tokenized_datasets["test"].shuffle(seed=64).select(range(100))

time: 52.4 ms (started: 2025-07-22 09:50:28 +08:00)


In [25]:
trainer.evaluate(small_test_dataset)

{'eval_loss': 0.8964799642562866, 'eval_accuracy': 0.63}

time: 1.91 s (started: 2025-07-22 09:50:34 +08:00)


### 保存模型和训练状态

- 使用 `trainer.save_model` 方法保存模型，后续可以通过 from_pretrained() 方法重新加载
- 使用 `trainer.save_state` 方法保存训练状态

In [26]:
trainer.save_model(model_dir)

time: 2.04 s (started: 2025-07-22 09:50:46 +08:00)


In [27]:
trainer.save_state()

time: 7.03 ms (started: 2025-07-22 09:51:56 +08:00)


In [23]:
# trainer.model.save_pretrained("./")

## Homework: 使用完整的 YelpReviewFull 数据集训练，看 Acc 最高能到多少